# DG v2 · B2 baseline + trọng số lớp

**Nhóm `baseline_weights`: baseline → class_weights. Mỗi thí nghiệm 20 epoch, khởi tạo B2 độc lập.** Hai notebook có thể chạy song song trên hai tài khoản; dùng cùng code ZIP, Cityscapes và seed42. Không sửa ARM/loss/model. Các mã V0–V3 cũ đã thay bằng tên mô tả.

**Hai input cho lượt mới: code ZIP `project-code-dg-v2.zip` mới + Cityscapes train2975.** Bật GPU/Internet và giữ Private. Không cần ACDC hay weights model cũ. Chạy tới INPUT CHECK PASS để xem dữ liệu/preview, rồi Save Version → Save & Run All.

Tập huấn luyện2380 và tập chọn model595 chia theo thành phố; danh sách có trong configs/dg_v2/splits. Chọn best bằng 0,5 clean-dev + 0,5 trung bình dev low-light/noise/haze cố định. Baseline có augmentation RGB cơ bản; hai thí nghiệm weather đổi sang mixture mở rộng.

Sau mỗi epoch committed, lưu best/last, optimizer/scheduler/scaler/RNG và ZIP nhóm này. Gần cửa sổ10giờ tự dừng; tải ZIP rồi dùng cùng notebook ở phiên mới với RESUME_INPUT. Hard-kill có thể bỏ qua finally: lấy ZIP hoàn chỉnh gần nhất. **Mỗi nhóm cần đủ hai thí nghiệm20/20; suite_complete chỉ nói nhóm này đã xong.** Chưa có kết quả GPU DG-v2; timing cần đo thực tế.

In [ ]:
from pathlib import Path
# Chỉ sửa các đường dẫn này nếu auto-find không rõ. Dùng cùng code ZIP cho hai tài khoản.
CODE_INPUT=Path('/kaggle/input')  # project-code-dg-v2.zip mới hoặc thư mục code đã giải nén
CITYSCAPES_ROOT=None             # Thư mục chứa leftImg8bit/train và gtFine/train; None = tự tìm
RESUME_INPUT=None                # Lượt đầu: None. Chạy tiếp: Path('...') đến kết quả CÙNG NHÓM


In [ ]:
GROUP='baseline_weights'
"""Standard-library source bootstrap shared by the two DG model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_dg_suite.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os,sys,time,json,subprocess
MARKERS=('scripts/run_dg_v2_suite.py','src/training/dg_v2.py',
    'configs/dg_v2/appearance.yaml','configs/dg_v2/splits/fit.csv','configs/dg_v2/splits/dev.csv')
MODEL='segformer_b2'
if not globals().get('_DG_BOOTSTRAP_DONE'):STARTED=time.monotonic()
BASE=Path('/kaggle/working')
WORK=BASE/f'dg_v2_{MODEL}_{GROUP}_seed42'
REPO=BASE/f'project_dg_v2_{GROUP}'
ARCHIVE=BASE/f'dg_v2_{MODEL}_{GROUP}_seed42.zip'
settings_identity=(str(CODE_INPUT),str(RESUME_INPUT),GROUP)
if globals().get('_DG_BOOTSTRAP_DONE'):
    assert settings_identity==_DG_BOOTSTRAP_SETTINGS,'Đã đổi input giữa phiên. Giữ ZIP rồi dùng phiên mới.'
    assert CITYSCAPES_ROOT is None or Path(CITYSCAPES_ROOT)==_DG_CITY_ROOT,'Đã đổi Cityscapes mount giữa phiên.'
    CITYSCAPES_ROOT=_DG_CITY_ROOT
    assert REPO.is_dir(),'Code folder missing; preserve outputs and use a fresh session'
    print('Dùng lại code và checkpoint trong phiên hiện tại.')
else:
    assert not WORK.exists() and not REPO.exists(),f'Thư mục đã có: {WORK}. Giữ kết quả trước; dùng phiên mới để đổi input.'
    if RESUME_INPUT is None:
        copy_source(resolve_source(CODE_INPUT,Path('/kaggle/input')),REPO)
    else:
        previous=Path(RESUME_INPUT)
        if previous.is_file():extract_safe(previous,WORK)
        else:
            found=[previous] if (previous/'suite_plan.json').is_file() else [p.parent for p in previous.rglob('suite_plan.json')]
            assert len(found)==1,'Set RESUME_INPUT to one DG-v2 result folder or ZIP'
            shutil.copytree(found[0],WORK)
        plan=json.loads((WORK/'suite_plan.json').read_text())
        assert (plan['protocol']=='cityscapes_groupdev_dg_v2' and plan['model']==MODEL
            and plan['seed']==42 and plan['group']==GROUP),'Sai nhóm kết quả. Dùng notebook và ZIP cùng nhóm.'
        copy_source(WORK/'source.zip' if (WORK/'source.zip').exists() else WORK/'source',REPO)
        import hashlib
        assert all(hashlib.sha256((REPO/n).read_bytes()).hexdigest()==h for n,h in plan['source_hashes'].items()),'Source changed'
        if CITYSCAPES_ROOT is not None:
            assert Path(CITYSCAPES_ROOT)==Path(plan['city_root']),'Resume requires the original Cityscapes mount'
        CITYSCAPES_ROOT=Path(plan['city_root'])
    if CITYSCAPES_ROOT is None:
        found=sorted({p.parent for p in Path('/kaggle/input').rglob('leftImg8bit')
            if (p/'train').is_dir() and (p.parent/'gtFine/train').is_dir()})
        assert len(found)==1,f'Set CITYSCAPES_ROOT explicitly: {found}'
        CITYSCAPES_ROOT=found[0]
    assert all((Path(CITYSCAPES_ROOT)/name).is_dir() for name in ['leftImg8bit/train','gtFine/train']),'Cityscapes train folders missing'
    _DG_BOOTSTRAP_SETTINGS=settings_identity
    _DG_CITY_ROOT=Path(CITYSCAPES_ROOT)
    _DG_BOOTSTRAP_DONE=True
os.chdir(REPO);sys.path.insert(0,str(REPO))
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
def run(*args):subprocess.run([sys.executable,'-u',*map(str,args)],check=True,cwd=REPO)
print('Cityscapes:',CITYSCAPES_ROOT,'Model:',MODEL,'Group:',GROUP)


In [ ]:
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow==11.3.0',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torch
assert torch.cuda.is_available(),'Enable GPU'
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2]))>=(2,6),'Require PyTorch>=2.6'
from src.experiments.run_support import snapshot_source
if WORK.exists() and not (WORK/'source.zip').is_file():snapshot_source(WORK/'source.zip',plan['source_hashes'])
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__)


## 1. Kiểm input và ảnh tăng cường
Chỉ đọc Cityscapes train; không đọc official-val/test hoặc ACDC.

In [ ]:
run('-m','scripts.run_dg_v2_suite','--city',CITYSCAPES_ROOT,'--root',WORK,
    '--archive',ARCHIVE,'--model',MODEL,'--group',GROUP,'--prepare-only')
print((WORK/'data/split.json').read_text())
print((WORK/'data/class_audit.csv').read_text())
from IPython.display import Image,display
display(Image(filename=str(WORK/'data/preview.png')))
print('INPUT CHECK PASS. Hai thí nghiệm trong nhóm:',json.loads((WORK/'suite_plan.json').read_text())['arms'])
print('Preview chứa ảnh dataset: giữ notebook Private.')


## 2. Tự chạy hai thí nghiệm: baseline → class_weights
Checkpoint/source/config/weights được khóa; không đổi nhóm giữa phiên.

In [ ]:
remaining=10-(time.monotonic()-STARTED)/3600
if remaining>0:
    run('-m','scripts.run_dg_v2_suite','--city',CITYSCAPES_ROOT,'--root',WORK,
        '--archive',ARCHIVE,'--model',MODEL,'--group',GROUP,'--session-hours',remaining)
else:print('Setup used session window. Preserve ZIP and resume in a fresh session.')


## 3. Tải ZIP của nhóm này
Gửi lại cả hai ZIP để kiểm tra và ghép bảng điểm source-dev. ACDC đánh giá sau khi chốt phương pháp.

In [ ]:
from IPython.display import FileLink,display
status=json.loads((WORK/'suite_status.json').read_text())
print('Group:',status['group'],'Status:',status['status'])
for job in status['jobs']:print(job)
print('Download:',ARCHIVE);display(FileLink(str(ARCHIVE)))
if status['status']!='suite_complete':print('Giữ ZIP; mở phiên mới cùng notebook, đặt RESUME_INPUT để chạy tiếp.')
